# __MODEL_LABEL_MARKDOWN__: weekly champion and challenger monitoring

Score the current champion and publish three refitted challengers using fresh data
and the champion's SQL baseline. Each run records four monitoring observations.

| Variant | Role and result |
| --- | --- |
| `STATIC_SCORE` | Score the current champion with its deployed coefficients and fitted settings. |
| `FROZEN_REFIT` | Publish a challenger with refitted coefficients and the saved basis and penalties. |
| `REESTIMATE_LAMBDA` | Publish a challenger with refitted coefficients and penalties using the saved basis. |
| `FULL_ADAPTIVE` | Publish a challenger after rebuilding the basis and refitting coefficients and penalties. |

This run does not promote a challenger. Review and explicitly promote a selected
package in `06_model_deployment.ipynb`.


## Configure monitoring.py

Edit the `monitoring.py` file beside this notebook. It owns the model identity,
connection settings, remote write guard, and `load_dataset()` function.
Replace the deliberate error in `load_dataset()` with your current source query
and enrichment. Return `PricingDataset` with its name, source, unique key, and
snapshot-date column. The commented SQL example uses a separate source runtime.
Keep the source columns required by the saved transforms and order rows consistently.

Notebook 07 and scheduled runs call the same `run()` function. This cell reloads
the module so edits take effect when you rerun it.


In [ ]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pricing_models.__PACKAGE_NAME__ import monitoring

monitoring = importlib.reload(monitoring)
print(f"Configuration: {monitoring.__file__}")


## Run and review

This writes four monitoring observations and three challenger packages to the
configured database. Confirm its destination and enable the remote write guard in
`monitoring.py` before an authorized run. Each observation is saved separately.
If a later operation fails, earlier observations and publications can remain;
repeating the same evidence reuses prior results.

In the runs table, `role` distinguishes the `CHAMPION` score from the three
`CHALLENGER` results. The `baseline_model_run_id`, `baseline_deployment_id`,
`baseline_rate_package_id` and `baseline_package_version` identify the champion
used for the comparison. Challenger rows also contain `model_run_id`,
`rate_package_id`, `package_version`, `model_version`, `package_status` and
`publication_reused`. The static score has no new candidate package.

Compare metrics, warnings and categorical drift before choosing a challenger for
review in 06. Exceptions keep their Python traceback in this notebook.


In [ ]:
report = monitoring.run()
display({"manifest_id": report.manifest_id})
print("Current champion and published challengers")
display(report.runs)
print("Metrics")
display(report.metrics)
print("Warnings and data checks")
display(report.issues)
print("Categorical drift")
display(report.drift)


## Run the same file on a schedule

The next cell prints the exact Python executable and file command for this project.
Use the project environment's kernel, then run the printed command once in a terminal
and review its log before scheduling it.

For Windows Task Scheduler, use the printed Python executable as **Program/script**
and the quoted `monitoring.py` path as **Add arguments**. For WSL cron, use the printed
command as the job command. Both can run from any working directory. Keep the project
path and interpreter fixed. The machine, WSL when used, source system, and database
must be available at run time.

Each script invocation prints its log path under `.local/monitoring_logs` and returns
a nonzero exit code on failure. This notebook does not register a scheduler task.


In [ ]:
import os
import shlex
import subprocess

script_path = Path(monitoring.__file__).resolve()
command = [sys.executable, str(script_path)]
print(subprocess.list2cmdline(command) if os.name == "nt" else shlex.join(command))
print(f"Python executable: {sys.executable}")
print(f'File argument: "{script_path}"')
print(f"Logs: {monitoring.MODEL_DIR / '.local' / 'monitoring_logs'}")
